# ⚡ GCP BigLake REST Catalog PySpark & Spark SQL Query Guide

이 주피터 노트북은 GCP **BigLake REST Catalog**(`https://biglake.googleapis.com/iceberg/v1/restcatalog`)에 적재된 Apache Iceberg 테이블(`default.external_weblog`)을 **PySpark 및 Spark SQL**을 사용하여 직접 조회 및 분석하는 엔터프라이즈 통합 가이드입니다.

### 🎯 주요 특징
1. **Zero Hardcoded Environment**: 하드코딩된 개인화/특정 프로젝트 문자열 없이 사용자 환경 변수(`GCP_PROJECT_ID`, `GCS_BUCKET_NAME`) 및 `google.auth.default()`를 통해 동적으로 자동 감지합니다.
2. **GCS Hadoop Connector Direct Integration**: ADC(`application_default_credentials.json`) 인증 정보를 바탕으로 Spark Hadoop GCS 커넥터를 자동 구성합니다.
3. **Optimized JVM & Spark Engine Parameters**: JDK 17/21+ 환경과의 완벽한 호환성을 위한 JVM 모듈 개방 플래그 및 Spark WholeStageCodegen 최적화 설정을 포함합니다.

In [ ]:
import os
import json
import google.auth
import google.auth.transport.requests

# 1. 시스템 JAVA_HOME 설정 (PySpark와 Java 26 간 호환성 오류 방지를 위해 JDK 21/17 지정)
possible_java_homes = [
    "/usr/lib/jvm/java-21-openjdk-amd64",
    "/usr/lib/jvm/java-17-openjdk-amd64",
    "/usr/lib/jvm/default-java"
]
for j_home in possible_java_homes:
    if os.path.exists(j_home):
        os.environ["JAVA_HOME"] = j_home
        os.environ["PATH"] = f"{j_home}/bin:" + os.environ.get("PATH", "")
        break

# 2. JDK 17/21+ 호환성을 위한 JVM Open Modules 설정
jvm_flags = (
    '--add-opens=java.base/java.lang=ALL-UNNAMED '
    '--add-opens=java.base/java.lang.invoke=ALL-UNNAMED '
    '--add-opens=java.base/java.lang.reflect=ALL-UNNAMED '
    '--add-opens=java.base/java.io=ALL-UNNAMED '
    '--add-opens=java.base/java.net=ALL-UNNAMED '
    '--add-opens=java.base/java.nio=ALL-UNNAMED '
    '--add-opens=java.base/java.util=ALL-UNNAMED '
    '--add-opens=java.base/java.util.concurrent=ALL-UNNAMED '
    '--add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED '
    '--add-opens=java.base/sun.nio.ch=ALL-UNNAMED '
    '--add-opens=java.base/sun.nio.cs=ALL-UNNAMED '
    '--add-opens=java.base/jdk.internal.ref=ALL-UNNAMED'
)
os.environ['JDK_JAVA_OPTIONS'] = jvm_flags

# 3. GCP Project ID 및 Bucket Name 동적 감지
credentials, auth_project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/cloud-platform']
)
if not credentials.valid:
    credentials.refresh(google.auth.transport.requests.Request())

PROJECT_ID = os.getenv("GCP_PROJECT_ID") or auth_project or "your-gcp-project-id"
BUCKET_NAME = os.getenv("GCS_BUCKET_NAME") or f"{PROJECT_ID}-bq-iceberg-demo-bucket"
TOKEN_STR = credentials.token or ""

# 4. GCS Connector Jar 및 ADC 경로 탐색 (상대 경로 사용으로 경로 내 쉼표 문자 분할 이슈 방지)
ADC_PATH = os.path.expanduser("~/.config/gcloud/application_default_credentials.json")
GCS_JAR = "gcs-connector-latest.jar"

with open(ADC_PATH) as f:
    adc_data = json.load(f)

CLIENT_ID = adc_data.get("client_id", "")
CLIENT_SECRET = adc_data.get("client_secret", "")
REFRESH_TOKEN = adc_data.get("refresh_token", "")

print(f"✅ JAVA_HOME     : {os.environ.get('JAVA_HOME')}")
print(f"✅ GCP Project ID : {PROJECT_ID}")
print(f"✅ GCS Bucket Name: gs://{BUCKET_NAME}")
print(f"✅ GCS Jar Path   : {GCS_JAR}")

In [ ]:
from pyspark.sql import SparkSession

CATALOG_NAME = "lakehouse_catalog"
WAREHOUSE_URI = f"bl://projects/{PROJECT_ID}/catalogs/{BUCKET_NAME}"
REST_URI = "https://biglake.googleapis.com/iceberg/v1/restcatalog"

spark = (
    SparkSession.builder
    .appName("BigLakeSparkSQLGuide")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.codegen.wholeStage", "false")
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1")
    .config("spark.jars", GCS_JAR)
    .config("spark.driver.extraClassPath", GCS_JAR)
    .config("spark.executor.extraClassPath", GCS_JAR)
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
    .config("spark.hadoop.fs.gs.auth.type", "USER_CREDENTIALS")
    .config("spark.hadoop.fs.gs.auth.client.id", CLIENT_ID)
    .config("spark.hadoop.fs.gs.auth.client.secret", CLIENT_SECRET)
    .config("spark.hadoop.fs.gs.auth.refresh.token", REFRESH_TOKEN)
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.rest.RESTCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", REST_URI)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", WAREHOUSE_URI)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.header.x-goog-user-project", PROJECT_ID)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.header.X-Iceberg-Access-Delegation", "vended-credentials")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.token", TOKEN_STR)
    .getOrCreate()
)

print(f"🚀 SparkSession initialized successfully!")
print(f"   Catalog Name : {CATALOG_NAME}")
print(f"   REST Catalog : {REST_URI}")

In [ ]:
print("--- 1. SHOW NAMESPACES ---")
spark.sql(f"SHOW NAMESPACES IN {CATALOG_NAME}").show()

print("\n--- 2. SHOW TABLES IN DEFAULT NAMESPACE ---")
spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.default").show()

In [ ]:
print("--- 3. DESCRIBE ICEBERG TABLE SCHEMA ---")
spark.sql(f"DESCRIBE TABLE {CATALOG_NAME}.default.external_weblog").show(truncate=False)

In [ ]:
print("--- 4. SPARK SQL GROUP BY AGGREGATION QUERY ---")
query_sql = f"""
    SELECT 
        event_type, 
        COUNT(*) AS total_events, 
        ROUND(SUM(amount), 2) AS total_amount, 
        ROUND(AVG(amount), 2) AS avg_amount 
    FROM {CATALOG_NAME}.default.external_weblog 
    GROUP BY event_type
    ORDER BY total_events DESC
"""
df_agg = spark.sql(query_sql)
df_agg.show()

In [ ]:
print("--- 5. SPARK SQL PARTITION PRUNING & FILTER QUERY ---")
pruning_sql = f"""
    SELECT 
        event_date,
        event_type, 
        COUNT(*) AS total_events, 
        ROUND(SUM(amount), 2) AS total_amount 
    FROM {CATALOG_NAME}.default.external_weblog 
    WHERE event_date BETWEEN '2026-07-03' AND '2026-07-05'
    GROUP BY event_date, event_type
    ORDER BY event_date ASC, total_events DESC
"""
df_pruning = spark.sql(pruning_sql)
df_pruning.show(15)

## 💡 Key Takeaways & Best Practices

1. **BigLake REST Catalog URL & Warehouse Format**:
   - REST Catalog URI: `https://biglake.googleapis.com/iceberg/v1/restcatalog`
   - Warehouse URI: `bl://projects/{PROJECT_ID}/catalogs/{BUCKET_NAME}`
2. **GCS Hadoop Connector Authentication**:
   - `spark.hadoop.fs.gs.impl`: `com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem`
   - `spark.hadoop.fs.gs.auth.type`: `USER_CREDENTIALS`
3. **Engine Optimization & Dynamic Security**:
   - JVM 17/21+ 호환성을 위해 `JDK_JAVA_OPTIONS` `--add-opens` 모듈 개방 필수.
   - WholeStageCodegen 조절 (`spark.sql.codegen.wholeStage: false`)을 통해 대규모 분산 쿼리 시 JVM JIT 시그니처 충돌 방지 및 안전한 분산 처리 보장.